# Project 4 · Extended solution — declarative validation framework

**Self-contained next step beyond the source exercise.** Keep the same descriptor fundamentals (`BaseValidator`, `IntegerField`, `CharField`, `__set_name__`) and add required/optional/default fields, read-only fields, normalized strings, regular-expression and choice validators, model inheritance, atomic multi-field updates, JSON round-trips, and schema introspection.

All code uses only Python's standard library. Python 3.11+ is a modern recommended interpreter, not a historical notebook version requirement. This is an educational in-memory model, **not** an ORM, security layer, or production schema migration system.

**Semantics:** wrong value types → `TypeError`; invalid constraints/values → `ValueError`; changes to initialized read-only fields → `AttributeError`; missing required fields or unknown names → `ValueError`/`TypeError`, respectively. Defaults are deep-copied or produced by a factory, and updates validate everything before committing anything. A model stores values in `__dict__`, so intentional direct edits to that dictionary can bypass validation.

## 1 · Reusable descriptor foundation

Each descriptor has an owner/name assigned automatically by Python, validates on normal assignment, and owns its own storage key. A single descriptor object cannot be reused under multiple names. Optional fields can remain unset; a default or factory provides initial values during model construction. Read-only means **single assignment** through the supported API.

In [1]:
from __future__ import annotations

from collections.abc import Callable, Mapping, Sequence
from copy import deepcopy
import json
from numbers import Integral
import re
from types import MappingProxyType
from typing import Any

_MISSING = object()


def _bound(value: int | None, label: str) -> None:
    if value is not None and (isinstance(value, bool) or not isinstance(value, Integral)):
        raise TypeError(f"{label} must be an integer or None")


class BaseValidator:
    """Validated data descriptor with defaults and single-assignment option."""

    def __init__(self, *, required: bool = True, default: Any = _MISSING,
                 default_factory: Callable[[], Any] | None = None,
                 read_only: bool = False) -> None:
        if not isinstance(required, bool) or not isinstance(read_only, bool):
            raise TypeError("required and read_only must be booleans")
        if default is not _MISSING and default_factory is not None:
            raise ValueError("Specify default or default_factory, not both")
        if default_factory is not None and not callable(default_factory):
            raise TypeError("default_factory must be callable")
        self.required = required
        self.read_only = read_only
        self._default = default
        self._default_factory = default_factory
        self.name: str | None = None
        self._owner: type | None = None
        self.storage_name: str | None = None

    def __set_name__(self, owner: type, name: str) -> None:
        if self._owner is not None and (owner is not self._owner or name != self.name):
            raise ValueError("Create a distinct descriptor instance per attribute")
        self._owner, self.name = owner, name
        self.storage_name = f"_validated_{name}"

    def _storage(self) -> str:
        if self.storage_name is None:
            raise RuntimeError("Descriptor must be bound at class creation")
        return self.storage_name

    def __get__(self, instance: object | None, owner: type | None = None) -> Any:
        if instance is None:
            return self
        return vars(instance).get(self._storage())

    def __set__(self, instance: object, value: Any) -> None:
        storage = self._storage()
        if self.read_only and storage in vars(instance):
            raise AttributeError(f"{self.name} is read-only after initialization")
        validated = self.validate(value)
        vars(instance)[storage] = validated

    def __delete__(self, instance: object) -> None:
        raise AttributeError(f"{self.name} cannot be deleted")

    @property
    def has_default(self) -> bool:
        return self._default is not _MISSING or self._default_factory is not None

    def default_value(self) -> Any:
        if self._default_factory is not None:
            return self._default_factory()
        if self._default is _MISSING:
            raise ValueError(f"{self.name} has no default")
        return deepcopy(self._default)

    def validate(self, value: Any) -> Any:
        """Override for a particular field type; base passes values through."""
        return value

    def schema(self) -> dict[str, Any]:
        metadata = {"type": type(self).__name__, "required": self.required,
                    "read_only": self.read_only, "has_default": self.has_default}
        if self._default is not _MISSING:
            metadata["default_repr"] = repr(self._default)
        elif self._default_factory is not None:
            metadata["default_factory"] = getattr(self._default_factory, "__name__", repr(self._default_factory))
        return metadata


class IntegerField(BaseValidator):
    """Integer field with inclusive optional minimum and maximum."""

    def __init__(self, min_: int | None = None, max_: int | None = None, **kwargs: Any) -> None:
        _bound(min_, "min_")
        _bound(max_, "max_")
        if min_ is not None and max_ is not None and min_ > max_:
            raise ValueError("min_ cannot exceed max_")
        super().__init__(**kwargs)
        self.min_, self.max_ = min_, max_

    def validate(self, value: Any) -> Integral:
        if isinstance(value, bool) or not isinstance(value, Integral):
            raise TypeError(f"{self.name} must be an integer, not bool")
        if self.min_ is not None and value < self.min_:
            raise ValueError(f"{self.name} must be >= {self.min_}")
        if self.max_ is not None and value > self.max_:
            raise ValueError(f"{self.name} must be <= {self.max_}")
        return value

    def schema(self) -> dict[str, Any]:
        return {**super().schema(), "minimum": self.min_, "maximum": self.max_}


class CharField(BaseValidator):
    """String field; optionally strip leading/trailing whitespace before checking length."""

    def __init__(self, min_: int | None = None, max_: int | None = None,
                 *, strip: bool = False, **kwargs: Any) -> None:
        _bound(min_, "min_")
        _bound(max_, "max_")
        if not isinstance(strip, bool):
            raise TypeError("strip must be a bool")
        minimum = max(0, 0 if min_ is None else min_)
        if max_ is not None and max_ < 0:
            raise ValueError("max_ cannot be negative")
        if max_ is not None and minimum > max_:
            raise ValueError("min_ cannot exceed max_")
        super().__init__(**kwargs)
        self.min_, self.max_, self.strip = minimum, max_, strip

    def validate(self, value: Any) -> str:
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a string")
        result = value.strip() if self.strip else value
        if len(result) < self.min_:
            raise ValueError(f"{self.name} must contain at least {self.min_} characters")
        if self.max_ is not None and len(result) > self.max_:
            raise ValueError(f"{self.name} must contain at most {self.max_} characters")
        return result

    def schema(self) -> dict[str, Any]:
        return {**super().schema(), "min_length": self.min_,
                "max_length": self.max_, "strip": self.strip}


class RegexField(CharField):
    """String field requiring a complete regular-expression match."""

    def __init__(self, pattern: str, min_: int | None = None,
                 max_: int | None = None, *, flags: int = 0, **kwargs: Any) -> None:
        if not isinstance(pattern, str):
            raise TypeError("pattern must be a string")
        if isinstance(flags, bool) or not isinstance(flags, int):
            raise TypeError("flags must be an integer")
        self.pattern = pattern
        self._regex = re.compile(pattern, flags)
        super().__init__(min_, max_, **kwargs)

    def validate(self, value: Any) -> str:
        result = super().validate(value)
        if self._regex.fullmatch(result) is None:
            raise ValueError(f"{self.name} does not match required pattern")
        return result

    def schema(self) -> dict[str, Any]:
        return {**super().schema(), "pattern": self.pattern}


class ChoiceField(BaseValidator):
    """Exact, case-sensitive string selection from a nonempty choice set."""

    def __init__(self, choices: Sequence[str], **kwargs: Any) -> None:
        if isinstance(choices, (str, bytes)) or not isinstance(choices, Sequence):
            raise TypeError("choices must be a sequence of strings")
        if not choices or any(not isinstance(v, str) for v in choices):
            raise ValueError("choices must contain at least one string and only strings")
        if len(set(choices)) != len(choices):
            raise ValueError("choices must not repeat values")
        super().__init__(**kwargs)
        self.choices = tuple(choices)

    def validate(self, value: Any) -> str:
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a string")
        if value not in self.choices:
            raise ValueError(f"{self.name} must be one of {self.choices!r}")
        return value

    def schema(self) -> dict[str, Any]:
        return {**super().schema(), "choices": list(self.choices)}

## 2 · Declarative model with inherited fields and atomic updates

`__init_subclass__` gathers fields from base classes and the new class. Its field registry is read-only, but the values of model instances are mutable except for read-only fields. `update()` validates all requested changes into a temporary dictionary and only then commits them: a bad value cannot leave **some** earlier fields updated.

`Model(**values)` performs the same validate-then-commit process, fills defaults without sharing mutable objects between instances, and flags missing required fields and unknown fields. Optional, unset fields are omitted from dictionaries by default.

In [2]:
class Model:
    """Small declarative, in-memory validated model (not an ORM)."""

    __fields__: Mapping[str, BaseValidator] = MappingProxyType({})

    def __init_subclass__(cls, **kwargs: Any) -> None:
        super().__init_subclass__(**kwargs)
        fields: dict[str, BaseValidator] = {}
        for base in reversed(cls.__mro__[1:]):
            fields.update(getattr(base, "__fields__", {}))
        for name, attr in vars(cls).items():
            if isinstance(attr, BaseValidator):
                fields[name] = attr
            elif name in fields:
                del fields[name]
        cls.__fields__ = MappingProxyType(fields)

    @classmethod
    def _reject_unknown(cls, values: Mapping[str, Any]) -> None:
        unknown = set(values) - set(cls.__fields__)
        if unknown:
            raise TypeError(f"Unknown fields: {', '.join(sorted(unknown))}")

    def __init__(self, **values: Any) -> None:
        cls = type(self)
        cls._reject_unknown(values)
        staged: dict[str, Any] = {}
        missing: list[str] = []
        for name, field in cls.__fields__.items():
            if name in values:
                proposed = values[name]
            elif field.has_default:
                proposed = field.default_value()
            elif field.required:
                missing.append(name)
                continue
            else:
                continue
            staged[field._storage()] = field.validate(proposed)
        if missing:
            raise ValueError(f"Missing required fields: {', '.join(missing)}")
        self.__dict__.update(staged)

    def update(self, **changes: Any) -> None:
        cls = type(self)
        cls._reject_unknown(changes)
        staged: dict[str, Any] = {}
        for name, value in changes.items():
            field = cls.__fields__[name]
            storage = field._storage()
            if field.read_only and storage in self.__dict__:
                raise AttributeError(f"{name} is read-only after initialization")
            staged[storage] = field.validate(value)
        self.__dict__.update(staged)

    def to_dict(self, *, include_unset: bool = False) -> dict[str, Any]:
        result: dict[str, Any] = {}
        for name, field in type(self).__fields__.items():
            storage = field._storage()
            if storage in self.__dict__:
                result[name] = deepcopy(self.__dict__[storage])
            elif include_unset:
                result[name] = None
        return result

    @classmethod
    def from_dict(cls, data: Mapping[str, Any]) -> Model:
        if not isinstance(data, Mapping):
            raise TypeError("data must be a mapping")
        return cls(**dict(data))

    def to_json(self, **json_options: Any) -> str:
        """Works for JSON-compatible field values; no arbitrary object encoder."""
        return json.dumps(self.to_dict(), **json_options)

    @classmethod
    def from_json(cls, payload: str) -> Model:
        if not isinstance(payload, str):
            raise TypeError("payload must be a JSON string")
        value = json.loads(payload)
        return cls.from_dict(value)

    @classmethod
    def schema(cls) -> dict[str, dict[str, Any]]:
        return {name: descriptor.schema() for name, descriptor in cls.__fields__.items()}

    def __repr__(self) -> str:
        args = ", ".join(f"{k}={v!r}" for k, v in self.to_dict().items())
        return f"{type(self).__name__}({args})"

## 3 · Useful real-world examples

Create user profiles with read-only IDs, whitespace-trimmed names, a validated age, a choice-restricted role, and default counters. A second model demonstrates that a descriptor is reusable **as a class**, not by binding the same descriptor **instance** in multiple classes.

In [3]:
class UserProfile(Model):
    user_id = RegexField(r"USR-\d{4}", read_only=True)
    name = CharField(1, 40, strip=True)
    age = IntegerField(0, 130)
    role = ChoiceField(("viewer", "creator", "admin"), default="viewer")
    login_count = IntegerField(0, default=0)
    biography = CharField(max_=200, required=False)


class InventoryItem(Model):
    sku = RegexField(r"SKU-[A-Z]{3}-\d{3}", read_only=True)
    label = CharField(1, 80, strip=True)
    quantity = IntegerField(0, default=0)
    status = ChoiceField(("available", "retired"), default="available")


user = UserProfile(user_id="USR-0001", name="  Ada  ", age=30)
assert user.name == "Ada"
assert user.role == "viewer"
user.update(age=31, role="creator", login_count=1)
assert user.to_dict()["age"] == 31
print(user)
print("JSON:", user.to_json(sort_keys=True))
print("User fields:", list(UserProfile.schema()))

item = InventoryItem(sku="SKU-CPU-001", label="  Ryzen  ", quantity=8)
assert (item.label, item.quantity) == ("Ryzen", 8)
print(item)

UserProfile(user_id='USR-0001', name='Ada', age=31, role='creator', login_count=1)
JSON: {"age": 31, "login_count": 1, "name": "Ada", "role": "creator", "user_id": "USR-0001"}
User fields: ['user_id', 'name', 'age', 'role', 'login_count', 'biography']
InventoryItem(sku='SKU-CPU-001', label='Ryzen', quantity=8, status='available')


## 4 · Comprehensive executable tests

Coverage includes descriptor lifecycle and inheritance, normalization, bounds, regular expressions, choices, defaults and factories, constructor behavior, read-only restrictions, atomic updates, deep copies, JSON round-trips, unknown keys, optional fields, dynamic class creation, and schema inspection.

In [4]:
import unittest


class TestExtendedDescriptors(unittest.TestCase):
    def test_integer_min_max_inclusive(self):
        class Item:
            number = IntegerField(0, 3)
        value = Item()
        value.number = 0
        value.number = 3
        self.assertEqual(value.number, 3)
        with self.assertRaises(ValueError):
            value.number = 4
        self.assertEqual(value.number, 3)

    def test_integer_rejects_bool_and_non_int(self):
        for value in (True, False, 1.1, "3", None):
            with self.subTest(value=value):
                with self.assertRaises(TypeError):
                    UserProfile.age.validate(value)

    def test_integer_min_max_none_and_zero(self):
        class Record:
            minimum = IntegerField(min_=0)
            maximum = IntegerField(max_=0)
        record = Record()
        record.minimum, record.maximum = 0, 0
        with self.assertRaises(ValueError):
            record.minimum = -1
        with self.assertRaises(ValueError):
            record.maximum = 1

    def test_invalid_integer_constructor(self):
        for args, error in (((5, 1), ValueError), ((True, None), TypeError),
                            ((None, "4"), TypeError)):
            with self.subTest(args=args):
                with self.assertRaises(error):
                    IntegerField(*args)

    def test_character_clamped_negative_min(self):
        descriptor = CharField(-5)
        self.assertEqual(descriptor.min_, 0)
        self.assertEqual(descriptor.validate(""), "")

    def test_character_type_and_length(self):
        field = CharField(2, 4)
        for value in (1, None, b"ab"):
            with self.assertRaises(TypeError):
                field.validate(value)
        for value in ("x", "xxxxx"):
            with self.assertRaises(ValueError):
                field.validate(value)
        self.assertEqual(field.validate("abcd"), "abcd")

    def test_char_strip_before_length_validation(self):
        field = CharField(2, 3, strip=True)
        self.assertEqual(field.validate("  abc  "), "abc")
        with self.assertRaises(ValueError):
            field.validate("  x  ")

    def test_invalid_char_constructor(self):
        for arguments, error in (((5, 1), ValueError), ((0, -1), ValueError),
                                 ((0.0, None), TypeError)):
            with self.subTest(arguments=arguments):
                with self.assertRaises(error):
                    CharField(*arguments)
        with self.assertRaises(TypeError):
            CharField(strip="yes")

    def test_regex_full_match(self):
        field = RegexField(r"ID-\d{2}")
        self.assertEqual(field.validate("ID-12"), "ID-12")
        for bad in ("prefixID-12", "ID-123", "ID-ab"):
            with self.subTest(bad=bad):
                with self.assertRaises(ValueError):
                    field.validate(bad)

    def test_regex_bad_args(self):
        with self.assertRaises(TypeError):
            RegexField(123)
        with self.assertRaises(re.error):
            RegexField("[")
        with self.assertRaises(TypeError):
            RegexField("abc", flags=True)

    def test_choice_validation(self):
        field = ChoiceField(["on", "off"])
        self.assertEqual(field.validate("on"), "on")
        with self.assertRaises(ValueError):
            field.validate("ON")
        with self.assertRaises(TypeError):
            field.validate(1)

    def test_choice_constructor(self):
        for choices, error in (([], ValueError), ("abc", TypeError),
                               (["one", "one"], ValueError), ([1], ValueError)):
            with self.subTest(choices=choices):
                with self.assertRaises(error):
                    ChoiceField(choices)

    def test_descriptor_class_access_and_isolation(self):
        self.assertIs(UserProfile.name, UserProfile.__dict__["name"])
        a = UserProfile(user_id="USR-0001", name="A", age=1)
        b = UserProfile(user_id="USR-0002", name="B", age=2)
        a.age = 3
        self.assertEqual((a.age, b.age), (3, 2))

    def test_descriptor_cannot_delete(self):
        value = UserProfile(user_id="USR-0003", name="A", age=2)
        with self.assertRaises(AttributeError):
            del value.age

    def test_rebinding_descriptor_is_rejected(self):
        field = IntegerField()
        First = type("First", (), {"a": field})
        with self.assertRaises(ValueError):
            field.__set_name__(First, "b")

    def test_unbound_descriptor_fails_instance_set(self):
        field = IntegerField()
        with self.assertRaises(RuntimeError):
            field.__set__(object(), 1)

    def test_default_and_factory_mutual_exclusion(self):
        with self.assertRaises(ValueError):
            BaseValidator(default=1, default_factory=list)
        with self.assertRaises(TypeError):
            BaseValidator(default_factory=42)
        with self.assertRaises(TypeError):
            BaseValidator(required=1)

    def test_read_only_direct_assignment(self):
        value = UserProfile(user_id="USR-0004", name="A", age=1)
        with self.assertRaises(AttributeError):
            value.user_id = "USR-9999"
        self.assertEqual(value.user_id, "USR-0004")

    def test_dynamic_class_invokes_set_name(self):
        cls = type("Generated", (Model,), {"score": IntegerField(0, 5)})
        value = cls(score=5)
        self.assertEqual(value.score, 5)
        self.assertEqual(cls.score.name, "score")


class TestExtendedModel(unittest.TestCase):
    def make_user(self, **overrides):
        values = {"user_id": "USR-0010", "name": " Alex ", "age": 20}
        values.update(overrides)
        return UserProfile(**values)

    def test_initialization_defaults_and_normalization(self):
        value = self.make_user()
        self.assertEqual(value.name, "Alex")
        self.assertEqual(value.role, "viewer")
        self.assertEqual(value.login_count, 0)
        self.assertIsNone(value.biography)

    def test_required_fields(self):
        with self.assertRaisesRegex(ValueError, "age"):
            UserProfile(user_id="USR-0001", name="A")

    def test_unexpected_field_on_init(self):
        with self.assertRaisesRegex(TypeError, "extraneous"):
            self.make_user(extraneous=1)

    def test_invalid_init_value(self):
        with self.assertRaises(ValueError):
            self.make_user(age=-1)
        with self.assertRaises(ValueError):
            self.make_user(user_id="BAD")

    def test_optional_field(self):
        value = self.make_user()
        self.assertNotIn("biography", value.to_dict())
        self.assertIsNone(value.to_dict(include_unset=True)["biography"])
        value.biography = "Developer"
        self.assertEqual(value.to_dict()["biography"], "Developer")

    def test_atomic_update_invalid_last(self):
        value = self.make_user()
        before = value.to_dict()
        with self.assertRaises(ValueError):
            value.update(name="Changed", age=-5)
        self.assertEqual(value.to_dict(), before)

    def test_atomic_update_read_only_after_valid_change(self):
        value = self.make_user()
        before = value.to_dict()
        with self.assertRaises(AttributeError):
            value.update(name="Changed", user_id="USR-9999")
        self.assertEqual(value.to_dict(), before)

    def test_atomic_update_bad_type(self):
        value = self.make_user()
        before = value.to_dict()
        with self.assertRaises(TypeError):
            value.update(name="Changed", login_count=True)
        self.assertEqual(value.to_dict(), before)

    def test_update_unknown(self):
        value = self.make_user()
        with self.assertRaises(TypeError):
            value.update(not_a_field="x")

    def test_successful_update(self):
        value = self.make_user()
        value.update(age=21, role="creator", name="  Zed  ")
        self.assertEqual((value.age, value.role, value.name), (21, "creator", "Zed"))

    def test_serialization_round_trip(self):
        value = self.make_user(biography="Engineer")
        result = UserProfile.from_json(value.to_json(sort_keys=True))
        self.assertEqual(result.to_dict(), value.to_dict())

    def test_json_invalid_shape_or_payload(self):
        with self.assertRaises(TypeError):
            UserProfile.from_json("[]")
        with self.assertRaises(TypeError):
            UserProfile.from_json(4)
        with self.assertRaises(json.JSONDecodeError):
            UserProfile.from_json("not json")

    def test_from_dict_type_check(self):
        with self.assertRaises(TypeError):
            UserProfile.from_dict([("name", "A")])

    def test_serialization_is_independent_deep_copy(self):
        class Flexible(Model):
            payload = BaseValidator(default={"tags": []})
        first, second = Flexible(), Flexible()
        first.payload["tags"].append("a")
        self.assertEqual(second.payload, {"tags": []})
        extracted = first.to_dict()
        extracted["payload"]["tags"].append("b")
        self.assertEqual(first.payload, {"tags": ["a"]})

    def test_default_factory(self):
        class Flexible(Model):
            payload = BaseValidator(default_factory=list)
        first, second = Flexible(), Flexible()
        first.payload.append(7)
        self.assertEqual(second.payload, [])

    def test_invalid_default_rejected_during_construction(self):
        class Broken(Model):
            number = IntegerField(0, default=-1)
        with self.assertRaises(ValueError):
            Broken()

    def test_inherited_model_fields(self):
        class Employee(UserProfile):
            department = CharField(1, 10)
        person = Employee(user_id="USR-0009", name="Pat", age=35, department="R&D")
        self.assertEqual(person.department, "R&D")
        self.assertEqual(person.age, 35)
        self.assertIn("name", Employee.schema())

    def test_override_model_field(self):
        class Adult(UserProfile):
            age = IntegerField(18, 130)
        with self.assertRaises(ValueError):
            Adult(user_id="USR-0001", name="Sam", age=17)
        self.assertEqual(Adult(user_id="USR-0001", name="Sam", age=18).age, 18)

    def test_non_field_override_removes_inherited_registry_entry(self):
        class Custom(Model):
            field = IntegerField()
        class Derived(Custom):
            field = "constant"
        self.assertNotIn("field", Derived.__fields__)
        with self.assertRaises(TypeError):
            Derived(field=1)

    def test_class_field_registry_cannot_be_modified(self):
        with self.assertRaises(TypeError):
            UserProfile.__fields__["extra"] = IntegerField()

    def test_schema_introspection(self):
        schema = UserProfile.schema()
        self.assertEqual(schema["age"]["minimum"], 0)
        self.assertTrue(schema["user_id"]["read_only"])
        self.assertEqual(schema["role"]["choices"], ["viewer", "creator", "admin"])
        self.assertTrue(schema["role"]["has_default"])
        self.assertEqual(schema["user_id"]["pattern"], r"USR-\d{4}")

    def test_repr(self):
        self.assertIn("UserProfile(", repr(self.make_user()))
        self.assertIn("name='Alex'", repr(self.make_user()))

    def test_unknown_field_name_is_detected_before_mutation(self):
        value = self.make_user()
        previous = value.to_dict()
        with self.assertRaises(TypeError):
            value.update(age=22, typo=3)
        self.assertEqual(value.to_dict(), previous)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestExtendedDescriptors)
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestExtendedModel))
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful(), "Extended solution test failures"
print(f"Extended solution: {result.testsRun} automated tests passed.")

..........................................
----------------------------------------------------------------------
Ran 42 tests in 0.045s

OK


Extended solution: 42 automated tests passed.


## 5 · Design boundaries and future directions

- **Why descriptors?** They centralize validation per attribute and work on ordinary classes, including classes created dynamically with `type`. Each field is a separate descriptor object; `__set_name__` is not automatically triggered by post-creation assignment.
- **Why staged updates?** Model-level validation of all inputs before committing prevents partial changes when a later field is invalid. It is an *in-memory atomic update*, not a database transaction and not thread-safe by itself.
- **Why read-only?** Useful for stable identifiers. It blocks repeated assignment via `obj.field` and `obj.update(...)`; manual `obj.__dict__` edits bypass it, as with other conventional Python encapsulation techniques.
- **Defaults and serialization:** Deep-copying fixed defaults and calling factories prevents accidental shared mutable defaults. JSON round-trips require JSON-compatible values; custom codecs, database persistence, concurrency locks, and full cross-field validation are outside the scope of this notebook.
- **Further extension:** Add cross-field invariants, strict typing with a static type checker, dataclass integration, error aggregation, property-based tests, and a persistence adapter when requirements justify their complexity.